# Space Debris Analysis Project


##### Phase 1: Problem Definition & Data Ingestion

* **Goal:** Define the objective and load raw data from its source without changing it.
* **Key Actions:**
  * Download data from APIs or files using `requests`
  * Load data into a DataFrame and review it with `df.head()` and `df.info()`



##### Phase 2: Data Cleaning & Preprocessing

* **Goal:** Fix errors and inconsistencies to make the dataset usable.
* **Key Actions:**
  * **Column Renaming:** Standardize column names by removing spaces and special characters
  * **Data Type Casting:** Convert columns to correct types (e.g., dates from `object` to `datetime64`)
  * **Duplicate Removal:** Find and remove duplicate rows
  * **Whitespace Cleaning:** Remove leading and trailing spaces from text
  * **Missing Values:** Decide to drop, fill, or keep missing values as appropriate



##### Phase 3: Exploratory Data Analysis (EDA)

* **Goal:** Find patterns, distributions, and relationships in the data using statistics and visualizations.
* **Key Actions:**
  * **Univariate Analysis:** Examine distribution of individual columns (e.g., histograms, value counts)
  * **Bivariate Analysis:** Compare relationships between variables (e.g., altitude vs. object type)
  * **Correlation Matrices:** Check how numerical variables relate to each other



##### Phase 4: Feature Engineering

* **Goal:** Create new variables from existing columns to improve analysis or modeling.
* **Key Actions:**
  * **Date Features:** Extract year, month, or calculate duration between dates
  * **Binning:** Group continuous variables into categories (e.g., Low vs. High orbit)
  * **Combined Features:** Create new metrics by combining existing ones



##### Phase 5: Modeling & Advanced Analysis (Optional)

* **Goal:** Apply machine learning or statistical models for predictions and insights.
* **Key Actions:** Split data, scale features, train models, and evaluate performance



##### Phase 6: Reporting & Deployment

* **Goal:** Share insights with stakeholders or create dashboards.
* **Key Actions:** Export clean data, build dashboards, or present findings




## Phase 1: Data Extracting and Saving the File

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

### Setting up Options

In [0]:
%skip
# Check your full dataset
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df)


#### Note: HTTP Response Codes & Methods

**Response Codes Reference**

* **1xx**: Informational (request received, continuing process)
* **200**: Success
* **3xx**: Redirection (further action needed)
* **401**: Unauthorized
* **403**: Forbidden
* **404**: Not Found

**Diagnostic Code Checks**

* `r.status_code`
* `r.request.headers`
* `r.request.body`
* `r.headers`

**When to Use Different Response Methods**

* **`.json()`** - Use when API returns JSON data
  * Most modern REST APIs (GitHub, weather, etc.)
  * Common for web services and public APIs

* **`.text`** - Use for plain text responses
  * HTML pages
  * Raw string data

* **`.content`** - Use for binary data
  * Images
  * File downloads
  * Returns raw bytes that you can save to a file

In [0]:
%skip
# data extracted url
url = "https://celestrak.org/pub/satcat.csv"

# path where you want to save the file
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw"
# path = "/Workspace/Users/guruvendra47@gmail.com/space-debris-project/space_debris_raw.csv" when you want code simpler and dont want filname seperated variable
# file name which you want to give
filename = "space_debris_raw.csv"
# combine full path with your filename
full_path = f"{path}/{filename}"


try:
    # getting data from url
    response = requests.get(url)

    # check if it is success or failed
    if response.status_code == 200:
        with open(full_path, "wb") as f:
            f.write(response.content)
            print(f"File saved successfully to: {full_path}")
    else:
        print(f"Failed to download. Status code: {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")


## Phase 2: Data Loadig & Initial Exploration & Formatting

In [0]:
path = "/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/data/raw/space_debris_raw.csv"
df = pd.read_csv(path)
df

### Basic Pandas Operations
- .head()
- .tail()
- .shape
- .columns
- .info()
- .describe()
- .rename(column=oldcolumnname, newcolumnname)

In [0]:
# head
df.head(10)

In [0]:
df.tail(10)

In [0]:
df.shape

In [0]:
# df.info()
df.dtypes

In [0]:
df.columns

#### Notes
**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

In [0]:
# rename the columns
rename_mapping={
        "OBJECT_NAME": "ObjectName",
        "OBJECT_ID": "ObjectID",
        "NORAD_CAT_ID": "CatalogID",
        "OBJECT_TYPE": "ObjectType",
        "OPS_STATUS_CODE": "OperationalStatus",
        "OWNER": "Owner",
        "LAUNCH_DATE": "LaunchDate",
        "LAUNCH_SITE": "LaunchSite",
        "DECAY_DATE": "DecayDate",
        "PERIOD": "OrbitalPeriodMin",
        "INCLINATION": "InclinationDegrees",
        "APOGEE": "MaxAltitudeKM",
        "PERIGEE": "MinAltitudeKM",
        "RCS": "RadarSizeSQM",
        "DATA_STATUS_CODE": "DataStatus",
        "ORBIT_CENTER": "OrbitCenter",
        "ORBIT_TYPE": "OrbitState",
    }

df = df.rename(columns=rename_mapping)

print(df.columns.tolist())

In [0]:
df.head(4)

### Understanding the Data
#### Space Debris Dataset Column Descriptions

* **`satellite_name`**: This columns contain names of satellite, rocket stage, or space debris.
* **`satellite_id`**: This is Unique No assigned to satellite, rocket stage, or space debris in International Designator (COSPAR ID) format as `YYYY-NNNAA` (Launch Year, Launch Number, Piece of launch).
* **`norad_cat_id`**: Unique tracking catalog number assigned by USSPACECOM/NORAD.
* **`satellite_type`**: Classification of the object:
  * `PAY`: Payload (active or inactive satellite).
  * `R/B`: Rocket Body (spent upper stage left in orbit).
  * `DEB`: Debris (fragmentation or dropped hardware).
* **`operation_status_code`**: Operational status indicator:
  * `+`: Operational/Active.
  * `-`: Non-operational/Inactive.
  * `P` : Partially Operational / Standby
  * `D`: Decayed (re-entered Earth's atmosphere).
  * `NaN`: Unknown or unassigned status.
* **`country`**: Country, space agency, or commercial owner.
* **`launch_date`**: Date the object was launched into space.
* **`launch_location`**: launch site/spaceport.
* **`decay_date`**: Date in which the object re-entered Earth's atmosphere. Missing values (`NaN`/`NaT`) mean the object is still in orbit.
* **`period_time_min`**: Orbital period in minutes (time required to complete one full orbit around Earth).
* **`Inclination`**: Angle (in degrees) between the orbital plane and Earth's equator.
* **`highest_altitude_km`**: Highest point of the orbit from Earth's surface (Apogee in km).
* **`lowest_altitude_km`**: Lowest point of the orbit from Earth's surface (Perigee in km).
* **`radar_cross_section`**: Radar Cross Section (\text{m}^2), representing the reflective physical size detected by radar.
* **`data_status_code`**: Administrative code indicating tracking data reliability.
* **`orbit_center`**: Body being orbited (`EA` = Earth) etc.
* **`orbit_type`**: Orbital state (`ORB` = Currently orbiting, `IMP` = Impacted/Decayed).



## Phase 3. Data Cleaning and Wrangling

### Step 1: Finding and Removing Duplicates
1. Identify duplicate rows  in the dataset.
2. Use suitable techniques to remove duplicate rows and verify the removal.
3. Summarize how to handle missing values appropriately.
4. Use ConvertedCompYearly to normalize compensation data.

In [0]:
# df.duplicated().value_counts()
df.duplicated().sum()

# to see duplicate
df[df.duplicated()]

# df.duplicated(subset=columnslist, keep=False) in order to check how many same values are there over the data 

# Removing the Duplicates
#df.drop_duplicates()

#### Note: No Duplicate Values Found

### Step 2: Removing Whitespaces
##### Note: 
- We only check string columns (dtype == 'object') because numeric columns (int, float) cannot contain leading/trailing whitespaces.

In [0]:
# step 1: Remove whitespaces

# Before: Count rows with whitespaces in each column
print("=== Before: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht}")
        else:
            print(f"{i}: {cnt_wht}")

print("\nRemoving....")
for i in df.columns:
    if df[i].dtype == 'object':
        df[i] = df[i].str.strip()

# After: Count rows with whitespaces in each column
print("\n=== After: Whitespace Analysis ===")
for i in df.columns:
    if df[i].dtype == 'object':
        wht_spc = (df[i].notna()) & (df[i] != df[i].str.strip())
        cnt_wht = wht_spc.sum()
        if cnt_wht > 0:
            print(f"{i}: {cnt_wht} ")
        else:
            print(f"{i}: {cnt_wht}")


### Step 3: Changing Data Types
- check which columns data type is wrong 
- change into "object", "int64", float64", "datatime64[ns]"
- we found that LaunchDate and DecayDate both columns dtypes need to change from object to Datetime

In [0]:
# check which columns has wrong data type
print("=== Before ===")
display(df.info())

print("\nChanging....")
df["LaunchDate"] = pd.to_datetime(df["LaunchDate"],format="mixed", errors= "coerce")
df["DecayDate"] = pd.to_datetime(df["DecayDate"],format="mixed", errors="coerce")

print("\n\n=== After ===")
print(df.dtypes)


### Step 4: Finding and Handling Missing Values
1. Identifying the missing values and converting in percentage for better view
2. Visualize missing values using a heatmap
3. Count the number of missing rows for a specific column (.value_counts())
4. Identifing the most frequent or more time repeated value in a spcific column
5. Taking decision to drop the rows or column or fill with mean, median, .idxmax()(most repeated value), .ffill()(forward-fill is filling with next value)
6. Visualize the distribution of a column after imputationF

In [0]:
# identifing the missing values
msgvalue = df.isnull().sum()

# In percentage for better view and rounded to 2 decimal 
prt = (msgvalue / len(df)*100).round(2)

# putting side by side for better view
dic = pd.DataFrame({"Missingvalues": msgvalue, "Percentage (%)":prt})
print(dic)

#### Note: Missing Values Summary
- OperationalStatus
- DecayDate
- OrbitalPeriodMinutes
- InclinationDegrees
- ApogeeKM
- PerigeeKM 
- RadarCrossSectionSQM
- DataStatusCode


#### Column: OperationalStatus
#### Column: OperationalStatus

Missing values occur because space tracking agencies only assign status codes to active/inactive satellites. used upper rocket bodies and debris do not have operational missions, so their status is naturally left blank so we filling that as `"Unknown"`.

In [0]:
# fillna NaN with Unknown for operation status code
df["OperationalStatus"] = df["OperationalStatus"].fillna("Unknown")
print(df.isnull().sum())

#### Column: DecayDate
#### Column: DecayDate
 - A missing `DecayDate` (`NaT`) means the object has not re-entered Earth's atmosphere and is still currently in orbit. We keep these values empty to maintain the `datetime64` data type so date calculations do not break. In Power BI, null values will be used to filter and render all active objects around Earth.

In [0]:

# DecayDate kept as NaT - missing values mean the object is still in space. Since it's not a string, I left NaT rather than changing it to "in space orbit".


#### Columns: Orbital Parameters

**Columns:** OrbitalPeriodMin, InclinationDegrees, MaxAltitudeKM, MinAltitudeKM

- Orbital parameters cannot be filled with zero (0 minutes = invalid orbit).
- In this case, we have to fill NaN values with median values, grouped by `ObjectType` and filtered by `OrbitCenter`.
- This ensures Earth-orbiting payloads use Earth payload medians, etc.

In [0]:
df["OrbitCenter"].unique()

In [0]:

# columns : period_time_min, Inclination, highest_altitude_km, lowest_altitude_km

cols = ["OrbitalPeriodMin", "InclinationDegrees", "MaxAltitudeKM", "MinAltitudeKM"]


orb = df["OrbitCenter"].unique()
# filtering data
for i in orb:
    mask = df["OrbitCenter"] == i
    # grouping the data
    for j in cols:
        df.loc[mask, j] = df.loc[mask, j].fillna(df[mask].groupby("ObjectType")[j].transform("median"))

print("\n=== Remaining Missing Values ===")
print(df.isnull().sum())

#### Note: Remaining Nulls values After Group Median
- After filtering with OrbitCenter and groupby ObjectType, 67 nulls remain in the "OrbitalPeriodMinutes", "OrbitalTiltDegrees", "HighestPointKm", and "LowestPointKm" columns because the mean could not be calculated because these columns value is null.
- As orbital data, making group medians NaN. We fill them with -1 (sentinel value) to keep numeric dtypes, preserve row count.

In [0]:
# 67 nan we will with sentinel value -1 as it non-earth orbit

df[cols] = df[cols].fillna(-1)
print(df.isnull().sum())


#### Column: RadarCrossSectionSQM
#### Note: RadarSizeSQM Null Handling

- Radar Cross Section (RCS) is missing for objects that are too small or far away to be measured by ground radar. We fill missing values with `0.0` to treat them as untracked physical sizes without breaking numeric models.

In [0]:
# radar_cross_section column
df["RadarSizeSQM"] = df["RadarSizeSQM"].fillna(0.0)
print(df.isnull().sum())


#### Column: DataStatus 

- According to space tracking documentation (CelesTrak SATCAT Documentation), 
- the official meanings of DataStatusCodes are:
   - NIE: No Initial Elements (Sensors detected the object at launch, but stable initial orbital calculations could not be established)
   - NEA: No Elements Available (Tracking elements are missing or discontinued)
   - NCE: No Current Elements (Historical tracking exists, but current active updates are unassigned)
   - NaN (Blank): Nominal Tracking or Active Elements Available (The standard baseline state for cataloged objects)

- Strategy: Replace codes with descriptive labels and fill NaN with "Active Tracking"

In [0]:
# 98 percentage is active tracking data which is null 

stcd = {"NIE":"No Initial Elements", "NEA":"No Elements Available", "NCE": "No Current Elements"}
df["DataStatus"] = df["DataStatus"].replace(stcd)
df["DataStatus"] = df["DataStatus"].fillna("Active Tracking")
print(df.isnull().sum())
df

## Step 5: Data Standardization

- Standardization is the process of transforming data into a common format, allowing the researcher to make the meaningful comparison

#### Columns to Standardize:

- ObjectType 
- OperationalStatus
- Owner 
- LaunchSite
- OrbitCenter
- OrbitState
- DataStatus.



### Column Modification Methods

**Column Headers**

* **`rename()`**: Changes header titles.

**Row Values**

* **`replace()`**: Use for quick code without a loop.
* **`map().fillna()`**: Use with a loop for max speed.

### .map() vs .replace()

**Use `.map()`:**
- When remapping an entire column into a new scale
- Unlisted values automatically turn to NaN
- Best for complete transformations

**Use `.replace()`:**
- When fixing a couple of specific values
- Everything else stays as-is
- Best for targeted replacements

### Columns: ObjectType & OperationalStatus

In [0]:

# No 1
# Column: ObjectType
dic = {"R/B":"Rocket Body", "PAY":"Payload", "DEB": "Debris"}
df["ObjectType"] = df["ObjectType"].replace(dic)
df.head()

# No2
# Column: OperationalStatus
dic = {"D": "Decayed (D)", "+": "Operational (+)", "-":"Non-Operational (-)", "P": "Partially Operational (P)"  }
df["OperationalStatus"] = df["OperationalStatus"].replace(dic)
df



### Column: Owner

In [0]:
# check Unique Country code
df["Owner"].unique()

In [0]:
owners = {
    "AB": "ARABSAT (Arab Satellite)",
    "AC": "Asia-Pacific Space Cooperation",
    "ABS": "ABS (Asia Broadcast Satellite)",
    "ALG": "Algeria",
    "ANG": "Angola",
    "ARGN": "Argentina",
    "ARM": "Armenia",
    "ASRA": "AsiaSat Telecommunications",
    "AUS": "Australia",
    "AZER": "Azerbaijan",
    "BEL": "Belgium",
    "BELA": "Belarus",
    "BGD": "Bangladesh",
    "BHR": "Bahrain",
    "BHUT": "Bhutan",
    "BOL": "Bolivia",
    "BRAZ": "Brazil",
    "BUL": "Bulgaria",
    "BWA": "Botswana",
    "CA": "Canada",
    "CHBZ": "China / Brazil (CBERS)",
    "CHLE": "Chile",
    "CIS": "Russia / Former USSR",
    "COL": "Colombia",
    "CRI": "Costa Rica",
    "CZCH": "Czech Republic",
    "DEN": "Denmark",
    "DJI": "Djibouti",
    "ECU": "Ecuador",
    "EGYP": "Egypt",
    "ESA": "European Space Agency",
    "ESRO": "European Space Research Org",
    "EST": "Estonia",
    "ETH": "Ethiopia",
    "EUME": "EUMETSAT",
    "EUTE": "EUTELSAT",
    "FGER": "West Germany (Former)",
    "FIN": "Finland",
    "FR": "France",
    "FRIT": "France / Italy",
    "GER": "Germany",
    "GHA": "Ghana",
    "GLOB": "Globalstar (Commercial)",
    "GREC": "Greece",
    "GRSA": "South Africa",
    "GUAT": "Guatemala",
    "HRV": "Croatia",
    "HUN": "Hungary",
    "IM": "Inmarsat (Commercial)",
    "IND": "India",
    "INDO": "Indonesia",
    "IRAN": "Iran",
    "IRAQ": "Iraq",
    "IRL": "Ireland",
    "ISRA": "Israel",
    "ISS": "International Space Station",
    "IT": "Italy",
    "ITSO": "INTELSAT (Commercial)",
    "JOR": "Jordan",
    "JPN": "Japan",
    "KAZ": "Kazakhstan",
    "KEN": "Kenya",
    "KWT": "Kuwait",
    "LAOS": "Laos",
    "LKA": "Sri Lanka",
    "LTU": "Lithuania",
    "LUXE": "Luxembourg",
    "MA": "Morocco",
    "MALA": "Malaysia",
    "MCO": "Monaco",
    "MDA": "Moldova",
    "MEX": "Mexico",
    "MMR": "Myanmar (Burma)",
    "MNE": "Montenegro",
    "MNG": "Mongolia",
    "MUS": "Mauritius",
    "NATO": "NATO",
    "NETH": "Netherlands",
    "NICO": "New ICO",
    "NIG": "Nigeria",
    "NKOR": "North Korea",
    "NOR": "Norway",
    "NPL": "Nepal",
    "NZ": "New Zealand",
    "O3B": "O3b Networks / SES",
    "ORB": "Orbcomm (Commercial)",
    "PAKI": "Pakistan",
    "PERU": "Peru",
    "POL": "Poland",
    "POR": "Portugal",
    "PRC": "China",
    "PRY": "Paraguay",
    "QAT": "Qatar",
    "RASC": "RASCOMSTAR-QAF",
    "ROC": "Taiwan",
    "ROM": "Romania",
    "RP": "Philippines",
    "RWA": "Rwanda",
    "SAFR": "South Africa",
    "SAUD": "Saudi Arabia",
    "SDN": "Sudan",
    "SEAL": "Sea Launch",
    "SEN": "Senegal",
    "SES": "SES S.A. (Commercial)",
    "SGJP": "Singapore / Japan",
    "SING": "Singapore",
    "SKOR": "South Korea",
    "SLB": "Solomon Islands",
    "SPN": "Spain",
    "STCT": "Singapore / Taiwan (ST-1)",
    "SVK": "Slovakia",
    "SVN": "Slovenia",
    "SWED": "Sweden",
    "SWTZ": "Switzerland",
    "TBD": "Unassigned / To Be Determined",
    "THAI": "Thailand",
    "TMMC": "Turkmenistan / Monaco",
    "TUN": "Tunisia",
    "TURK": "Turkey",
    "UAE": "United Arab Emirates",
    "UGA": "Uganda",
    "UK": "United Kingdom",
    "UKR": "Ukraine",
    "URY": "Uruguay",
    "US": "United States",
    "USBZ": "United States / Brazil",
    "VAT": "Vatican City",
    "VENZ": "Venezuela",
    "VTNM": "Vietnam",
    "ZWE": "Zimbabwe",
}

df["Owner"] = df["Owner"].replace(owners)

df

#### making copy of that file

In [0]:
df = df.copy()

### Column: LaunchSite 


In [0]:
df["LaunchSite"].unique()

In [0]:
sitemapping = {
    "AFETR": "United States (AFETR)",
    "AFWTR": "United States (AFWTR)",
    "CAS": "Spain (CAS)",
    "DLS": "Russia (DLS)",
    "ERAS": "United States (ERAS)",
    "FRGUI": "French Guiana (FRGUI)",
    "HGSTR": "Algeria (HGSTR)",
    "JJSLA": "South Korea (JJSLA)",
    "JSC": "China (JSC)",
    "KODAK": "United States (KODAK)",
    "KSCUT": "Japan (KSCUT)",
    "KWAJ": "Marshall Islands (KWAJ)",
    "KYMSC": "Russia (KYMSC)",
    "NSC": "South Korea (NSC)",
    "PLMSC": "Russia (PLMSC)",
    "RLLB": "New Zealand (RLLB)",
    "SCSLA": "China (SCSLA)",
    "SEAL": "International (SEAL)",
    "SEMLS": "Iran (SEMLS)",
    "SMTS": "Iran (SMTS)",
    "SNMLP": "Kenya (SNMLP)",
    "SRILR": "India (SRILR)",
    "SUBL": "International (SUBL)",
    "SVOBO": "Russia (SVOBO)",
    "TAISC": "China (TAISC)",
    "TANSC": "Japan (TANSC)",
    "TYMSC": "Kazakhstan (TYMSC)",
    "VOSTO": "Russia (VOSTO)",
    "WLPIS": "United States (WLPIS)",
    "WOMRA": "Australia (WOMRA)",
    "WRAS": "United States (WRAS)",
    "WSC": "China (WSC)",
    "XICLF": "China (XICLF)",
    "YAVNE": "Israel (YAVNE)",
    "YSLA": "China (YSLA)",
    "YUN": "North Korea (YUN)",
}

df["LaunchSite"] = df["LaunchSite"].map(sitemapping).fillna(df["LaunchSite"])
# or
df["LaunchSite"] = df["LaunchSite"].replace(sitemapping)
df.head(20)

### Column: OrbitCenter

In [0]:
df["OrbitCenter"].unique()

In [0]:
orbname = {
    # Planets & Celestial Bodies
    "ME": "Mercury (ME)",
    "VE": "Venus (VE)",
    "EA": "Earth (EA)",
    "MA": "Mars (MA)",
    "JU": "Jupiter (JU)",
    "SA": "Saturn (SA)",
    "UR": "Uranus (UR)",
    "NE": "Neptune (NE)",
    "PL": "Pluto (PL)",
    "SU": "Sun (SU)",
    "MO": "Moon (MO)",
    "CO": "Comet (CO)",
    # Systems & Lagrange Points
    "EM": "Earth-Moon System (EM)",
    "SS": "Solar System (SS)",
    "AS": "Asteroids (AS)",
    "EL": "Earth-Moon Lagrange (EL)",
    "EL1": "Earth-Moon L1 (EL1)",
    "EL2": "Earth-Moon L2 (EL2)",
    # Specific High-Profile Space Stations
    "25544": "ISS (25544)",
    "28358": "Tiangong Target (28358)",
    "48274": "Tianhe Core (48274)",
}

df["OrbitCenter"] = df["OrbitCenter"].replace(orbname)
df

### Column: OrbitState

In [0]:
df["OrbitState"].unique()

In [0]:
obste = {
    # Existing Array Codes
    "IMP": "Impact (IMP)",
    "ORB": "Orbiter (ORB)",
    "LAN": "Lander (LAN)",
    "DOC": "Dock / Rendezvous (DOC)",
    # Other CelesTrak Mission Types
    "FLY": "Flyby (FLY)",
    "SAMP": "Sample Return (SAMP)",
    "CREW": "Crewed (CREW)",
    "CARG": "Cargo / Resupply (CARG)",
    "DEP": "Deployer (DEP)",
    "REL": "Relay (REL)",
}

df["OrbitState"] = df["OrbitState"].replace(obste)
df

# Phase 4: Exploratory Data Analysis (EDA)
- In this phase, we analyze distributions, feature relationships, correlations, and historical launch trends across the cleaned space debris dataset.

**1. Univariate Analysis (One Variable)**

*Looking at columns individually to understand distributions, spot outliers, and check skewness.*

* **Histograms & Boxplots** — Use histograms (with KDE) to see the spread of numerical data; use boxplots to spot extreme outliers.
* **The `describe()` Check** — Run `df.describe().T` on numerical columns. It's a quick sanity check on mean, min, max, and quartiles.
* **Categorical Breakdown** — Run `.value_counts()` or plot bar charts / pie charts on text columns to see the volume of each category.

**2. Bivariate Analysis (Two Variables)**

*Exploring relationships between pairs of variables.*

* **Scatter Plots** — Best for seeing if one variable goes up when another goes down (e.g., perigee vs. apogee).
* **Box Plots by Category** — Compare a numerical variable across groups (e.g., orbital period by object type).
* **Correlation & Pivot Tables** — Run pairwise correlation to see which numerical columns move together. Use pivot tables to group data and find averages across two categories.

**3. Multivariate Analysis (Three+ Variables)**

*Adding a grouping or temporal dimension to uncover deeper patterns.*

* **Color-Coded Scatter Plots** — Take a scatter plot and color-code by a category (hue) to see deeper patterns.
* **Correlation Heatmaps** — Visualize multiple numerical variables at once to detect multicollinearity.
* **Time Series** — Plot metrics over time (e.g., launches per year by object type) to find trends, seasonal spikes, or accumulation patterns.
* **Domain-Specific Questions** — Answer the actual research questions (e.g., the ratio of debris to active satellites, top countries by launch volume).

## 4.1 Univariate Analysis



#### Numeric Variables
- Summary stats: count, mean, median, std, min, max, quartiles, IQR, skewness
- Visuals: histogram, boxplot, density plot
- Check for multimodality (multiple peaks) and heavy tails (outliers)

#### Categorical Variables
- Frequency counts
- Bar charts and Pareto charts
- Check rare categories (low-frequency levels) and cardinality

#### Date/Time Variables
- Check for time range, gaps, time granularity
- Visuals: time series plot, seasonality checks, heatmap (for example: day vs hour), bar chart (month/year)

#### Text Variables
- Token counts, most common words, length distributions
- Histogram of text lengths
- Word cloud / bar of top words

#### 4.1.a Numberic Variable

In [0]:
# Number Variable

num = df.select_dtypes(include=["number"])

summary = num.describe().T
summary["median"] = num.median()
# summary = summary.rename(columns={"50%": "median"})
summary["IQR"] = summary["75%"] - summary["25%"]
summary["skewness"] = num.skew()

# Additional statistics
# Calculate kurtosis to measure the presence of extreme outliers or heavy tails in numerical features
summary["Kurtosis"] = num.kurtosis()
summary["Missing values"] = num.isna().sum()

print("Numeric Summary Statistics:")
summary.round(2)




## Summary Findings from describe()

1. **Row Count**: The dataset contains 70,586 rows across all numerical columns, with no missing values after imputation.

2. **Outliers Present**: Standard deviation values are very high relative to the mean in several columns (especially `MaxAltitudeKM` with std = 167,604 km and `OrbitalPeriodMin` with std = 19,840 min), indicating the presence of extreme outliers.

3. **High Variability**: Standard deviation measures how far data points scatter from the mean. Large std values indicate wide spread and heterogeneous data distribution.

4. **Placeholder Values**: Minimum values of -1.0 in orbital columns (`OrbitalPeriodMin`, `InclinationDegrees`, `MaxAltitudeKM`, `MinAltitudeKM`) indicate missing data placeholders that need to be addressed.

5. **Right-Skewed Distributions**: Most orbital parameters show heavy positive skewness, meaning data is clustered near lower values with long tails extending to extreme highs (typical of space objects concentrated in LEO with few deep-space outliers).

6. **IQR Analysis**: The Interquartile Range (Q3 - Q1) shows that the middle 50% of satellites operate in relatively narrow bands, while outliers drive the high maximums and standard deviations. 

%md
KDE (Kernel Density Estimation) can only be used with continuous numerical data (like measurements, weights, altitudes, or speeds)—never with categories, text labels, or discrete counts.

**Charts Where KDE Can Be Used**

* Histograms (`sns.histplot`)
* Probability Density Plots (`sns.kdeplot`)
* Distribution Plots (`sns.displot`)

In [0]:
# Visuals: histogram, boxplot, density plot
# Check for multimodality (multiple peaks) and heavy tails (outliers)

numcols = list(df.select_dtypes(include=["number"]).columns)

fig, axes = plt.subplots(len(numcols), 3, figsize=(18, 4 * len(numcols)), squeeze=False)

for idx, i in enumerate(numcols):

    # Histogram + KDE
    sns.histplot(df[i].dropna(), bins=50, kde=True, color='teal', ax=axes[idx, 0])
    axes[idx, 0].set_title(f'Distribution: {i}')
    axes[idx, 0].set_xlabel(i)

    # Density Plot (KDE)
    sns.kdeplot(df[i].dropna(), fill=True, ax=axes[idx, 1], color="purple")
    axes[idx, 1].set_title(f'Density Plot: {i}')
    axes[idx, 1].set_xlabel(i)

    # Boxplot
    sns.boxplot(y=df[i].dropna(), color="orange", ax=axes[idx, 2])
    axes[idx, 2].set_title(f'Boxplot: {i}')
    axes[idx, 2].set_ylabel(i)

plt.tight_layout()
plt.show()


#### 4.1.b Categorical Variables
- Frequency counts
- Bar charts and Pareto charts
- Check rare categories (low-frequency levels) and cardinality

In [0]:
# Frequency Counts

catcols = df.select_dtypes(include=["object"])
lstcol = list(catcols.columns) # instead list(catcols.columns) you can use catcols.columns.tolist()

for i in lstcol:
    print(f"Column: {i}")
    freq = df[i].value_counts().head(20)
    prt = (freq/len(df)*100).round(2)
    dtfr = pd.DataFrame({"Frequency counts": freq, "Percentage": prt})
    print(f"{dtfr}\n")

### Categorical Variables — Bar Charts & Pareto Charts

**Matplotlib vs. Seaborn Plotting Syntax**

| Library | How it draws on axes | Example |
| --- | --- | --- |
| Matplotlib (OO) | Methods called directly on the axes object | `ax.bar(x, y)` or `ax.plot(x, y)` |
| Seaborn (wrapper) | Axes passed as a keyword argument | `sns.histplot(x, y, ax=ax)` |

Matplotlib needs **explicit numeric coordinates** (`rng = list(range(len(counts)))`) because its low-level functions only understand numbers for positioning. Seaborn handles categorical text labels natively.

**What this section does**
* Filters out high-cardinality columns (e.g. `ObjectName` ~29K, `ObjectID` ~70K) that would freeze matplotlib
* For each categorical column with **≤ 20 unique values**, creates two side-by-side panels:
  * **Left — Bar chart:** frequency counts with value labels
  * **Right — Pareto chart:** bars + cumulative % line with an 80% threshold

**Loop logic — for each categorical column:**

1. **Bar chart (left panel)** — Seaborn `barplot` with horizontal bars, `viridis` palette, count labels on each bar
2. **Pareto chart (right panel)** — Matplotlib bars using numeric positions (`rng`), steelblue colour, 80% alpha
3. **Cumulative % line** — Secondary axis (`twinx`), red line with markers, 80% threshold dotted line, percentage formatter on the right y-axis

In [0]:


catcol = df.select_dtypes(include=["object"])
lstcol = list(catcol.columns)
# lstcol = [i for i in catcols if df[i].nunique() <= 20]

fig, axes = plt.subplots(len(lstcol), 2, figsize=(12, 4 * len(lstcol)), squeeze=False)

for index, i in enumerate(lstcol):
    freq = df[i].value_counts().head(20)

    # Bar Chart
    axbar = axes[index, 0]
    sns.barplot(x=freq.values, y=freq.index, hue=freq.index, palette="viridis", ax=axbar)
    axbar.set_title(i)
    axbar.set_xlabel("count")
    axbar.set_ylabel(i)
    # adding the labels
    for y, x in enumerate(freq.values):
        axbar.text(x + freq.max() * 0.01, y, f'{x:,}', va='center', fontsize=9)

    # Pareto Chart
    rng = list(range(len(freq)))
    
    axpar = axes[index, 1]
    axpar.bar(rng, freq.values, color="steelblue", alpha=0.8)
    axpar.set_title(f"Pareto Chart: {i}")
    axpar.set_xlabel(i)
    axpar.set_ylabel("count")
    # mentioning the tickers
    axpar.set_xticks(rng)
    axpar.set_xticklabels(freq.index, rotation=45, ha="right", fontsize=9)

    # Line chart in bar chart as secondary axis
    prct = (freq/freq.sum() * 100)
    cmpt = prct.cumsum()
    axlin = axpar.twinx()
    axlin.plot(rng, cmpt.values, color="red", marker="o", linewidth=2, markersize=5)
    axlin.set_ylabel("cumulative %", color="red")
    axlin.set_ylim(0,110)

    # dash line
    axlin.axhline(y=80, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
    # marking
    from matplotlib.ticker import PercentFormatter
    axlin.yaxis.set_major_formatter(PercentFormatter())
    axlin.tick_params(axis="y", labelcolor="red")
    axlin.legend(loc="center right", fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nVisualized {len(lstcol)} categorical columns: {lstcol}")

skipped = [col for col in catcols if col not in lstcol]
if skipped:
    print(f"\nSkipped (high cardinality > 20): {skipped}")
    for col in skipped:
        print(f"   - {col}: {df[col].nunique():,} unique values")

#### 4.1.c Date/Time Variables
- Check for time range, gaps, time granularity
- Visuals: time series plot, seasonality checks, heatmap (for example: day vs hour), bar chart (month/year)

In [0]:
dtcols = list(df.select_dtypes(include=["datetime"]).columns)

for i in dtcols:
    print(i)
    mindate = df[i].min()
    maxdate = df[i].max()
    timespan = (maxdate - mindate).days
    print(f"date range from {mindate} to {maxdate}")
    print(f"timespan:{timespan} days and ({timespan/365.25:.2f}) years")

    withouttime = (df[i].dt.hour == 0).sum()
    withtime = (df[i].dt.hour != 0).sum()

    print(f"records withouttime {withouttime} ({withouttime/len(df)*100:.1f}%)" )
    print(f"records withtime {withtime} ({withtime/len(df)*100:.1f}%)" )

    print("\n3. GAP DETECTION:")

    df_sorted = df.sort_values(i)
    date_diffs = df_sorted[i].diff()

    print("Top 5 largest gaps:")
    for idx, gap in date_diffs.nlargest(5).dropna().items():
        print(f"- {gap.days:,} days (ending {df_sorted.loc[idx, i].date()})")

    large_gaps = date_diffs[date_diffs > pd.Timedelta(days=365)]
    print(f"\nGaps > 1 year: {len(large_gaps)}")
    for idx, gap in large_gaps.items():
        print(f"- {gap.days:,} days ending at {df_sorted.loc[idx, i].date()}")

    print("\n3. GAP DETECTION:")
    
    df_sorted = df.sort_values(i)
    date_diffs = df_sorted[i].diff()


    largest_gaps = date_diffs.nlargest(5)
    print("Top 5 largest gaps between consecutive records:")

    for idx, gap in largest_gaps.dropna().items():
        gap_days = gap.days
        end_date = df_sorted.loc[idx, i].date()
        print(f"{gap_days:,} days (ending {end_date})")

    large_gaps = date_diffs[date_diffs > pd.Timedelta(days=365)]
    if len(large_gaps) > 0:
        print(f"\nFound Gap{len(large_gaps)} gaps > 1 year:")
        for idx, gap in largest_gaps.dropna().items():
            gap_days = gap.days
            end_date = df_sorted.loc[idx, i].date()
            print(f"{gap_days:,} days (ending {end_date})")
    else:
        print("\n -> No Major Gaps (all gaps < 1 year)")



In [0]:
# Date/Time Variable Analysis

# select_dtypes() filters columns by data type
dtcol = df.select_dtypes(include=["datetime"])  # Get DataFrame with only datetime columns
dtlst = list(dtcol.columns)  # Convert column names to a list

print(f"Datetime columns found: {dtlst}\n")

# Loop through each datetime column and analyze it
for i in dtlst:
    print(f"\nColumn: {i}")

    # 1. Time Range Check
    print("\n1. TIME RANGE ANALYSIS")
    # Find the earliest and latest dates to understand data coverage
    mindate = df[i].min()  # Find the earliest date in the column
    maxdate = df[i].max()  # Find the latest date in the column
    timespan = (maxdate - mindate).days  # Calculate total days between earliest and latest
    print(f"Date Range for {i}: {mindate} to {maxdate}")
    print(f"Time span: {timespan} days ({timespan/365.25:.1f} years)")  
    
    
    


    # 2. Time Granularity Check (checking if timestamps contain time or just dates)
    print("\n2. TIME GRANULARITY: Does Date Columns contain time or just dates")
    # .dt.hour extracts the hour component from datetime
    # If hour == 0, it means the time is midnight (00:00:00) - likely date-only data
    withouttime = (df[i].dt.hour == 0).sum() 
    withtime = (df[i].dt.hour != 0).sum()
    print(f"Records without time component(00:00:00): {withouttime:,} ({withouttime/len(df)*100:.1f}%)")
    print(f"Records with time component: {withtime:,} ({withtime/len(df)*100:.1f}%)")
    if withouttime > len(df) * 0.99:
        print(f"-> Precision: DATE ONLY (no time component)")
    else:
        print(f"-> Precision: DATE + TIME")
    


    # 3. Gap Detection Check
    # Find periods with no data (missing dates) by checking time differences
    print("\n3. GAP DETECTION:")
    
    df_sorted = df.sort_values(i)
    date_diffs = df_sorted[i].diff()
    # .diff() calculates the difference between each row and the previous row

    # Find and display the 5 largest gaps
    largest_gaps = date_diffs.nlargest(5)

    print("Top 5 largest gaps between consecutive records:")
    for idx, gap in largest_gaps.dropna().items():
        gap_days = gap.days
        end_date = df_sorted.loc[idx, i].date()
        print(f"{gap_days:,} days (ending {end_date})")

    large_gaps = date_diffs[date_diffs > pd.Timedelta(days=365)]
    # Check for major data holes (gaps larger than 1 year)
    # pd.Timedelta(days=365) creates a timedelta object representing 1 year

    if len(large_gaps) > 0:
        print(f"\nFound Gap{len(large_gaps)} gaps > 1 year:")
        for idx, gap in largest_gaps.dropna().items():
            gap_days = gap.days
            end_date = df_sorted.loc[idx, i].date()
            print(f"{gap_days:,} days (ending {end_date})")
    else:
        print("\n No Major Gaps (all gaps < 1 year)")
    


    
    # 4. Yearly Distribution
    # Analyze how records are distributed across years
    print("\n4. YEARLY DISTRIBUTION:")
    
    # Extract year from datetime and count occurrences
    # .dt.year extracts the year component from datetime
    # .value_counts() counts how many records exist for each year
    # .sort_index() sorts by year (ascending order)
    year_counts = df[i].dt.year.value_counts().sort_index()

    busiest_year = year_counts.idxmax()
    slowest_year = year_counts.idxmin()
    
    print(f" Data spans {len(year_counts)} years: {year_counts.index.min()} to {year_counts.index.max()}")
    print(f" Average records per year: {len(df)/len(year_counts):.0f}")
    
    # Find the busiest and slowest years
    # .idxmax() returns the index (year) with the maximum count
    # .idxmin() returns the index (year) with the minimum count
    print(f" Peak year: {int(busiest_year)} with {year_counts.max():,} records")
    print(f" Slowest year: {int(slowest_year)} with {year_counts.min():,} records")
    
    
    
    # 5. Monthly Distribution
    # Analyze how records are distributed across months
    print("\n5. MONTHLY DISTRIBUTION:")
    
    # Extract month from datetime and count occurrences
    # .dt.month extracts the month component (1-12) from datetime

    month_counts = df[i].dt.month.value_counts().sort_index()
    
    print(f" Average records per month: {len(df)/12:.0f}")
    
    # Find the busiest and slowest months
    busiest_month = month_counts.idxmax()
    slowest_month = month_counts.idxmin()
    
    # Get month names dynamically
    import calendar
    # Convert to int (needed because dt.month returns float64 when column has NaT values)
    busiest_month_name = calendar.month_name[int(busiest_month)]
    slowest_month_name = calendar.month_name[int(slowest_month)]
    
    print(f" Peak month: {busiest_month_name} with {month_counts.max():,} records")
    print(f" Slowest month: {slowest_month_name} with {month_counts.min():,} records")

# Final summary
print(f"\n{'='*60}")
print("Analysis Complete")
print(f"{'='*60}")



In [0]:
# Chart 1: Time Series Plot (Yearly Trend)
# Get datetime columns
dtcol = df.select_dtypes(include=["datetime"])
dtlst = list(dtcol.columns)

print(f"Creating Time Series Plots for: {dtlst}\n")

for col in dtlst:
    print(f"Column: {col}")
    
    # Prepare data - remove NaT values
    valid_dates = df[col].dropna()
    
    # Extract years and count occurrences
    years = valid_dates.dt.year
    yrcounts = years.value_counts().sort_index()
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(16, 6))
    
    # Plot time series with filled area
    ax.plot(yrcounts.index, yrcounts.values, marker='o', linewidth=2, markersize=5, color='steelblue', label='Yearly Count')
    ax.fill_between(yrcounts.index, yrcounts.values, alpha=0.3, color='lightblue')
    # Styling
    ax.set_title(f'{col} - Time Series (Yearly Trend)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left', fontsize=10)
    
    # Add trend line
    z = np.polyfit(yrcounts.index, yrcounts.values, 1)
    p = np.poly1d(z)
    ax.plot(yrcounts.index, p(yrcounts.index), "r--", alpha=0.7, linewidth=2, label=f'Trend (slope={z[0]:.1f}/year)')
    ax.legend(loc='upper left', fontsize=10)
    
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # extra if you want you can delete    
    # Print statistics
    print(f" Statistics:")
    print(f"   • Total years: {len(yrcounts)}")
    print(f"   • Peak year: {yrcounts.idxmax()} ({yrcounts.max():,} records)")
    print(f"   • Lowest year: {yrcounts.idxmin()} ({yrcounts.min():,} records)")
    print(f"   • Average per year: {yrcounts.mean():.0f}")
    print(f"   • Trend: {'+' if z[0] > 0 else ''}{z[0]:.1f} records/year\n")



In [0]:
# Chart 2: Seasonality Decomposition - Trend, Seasonal, Residual
import matplotlib.pyplot as plt
import calendar
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Get datetime columns
dtcol = df.select_dtypes(include=["datetime"])
dtlst = list(dtcol.columns)

print(f"Creating Seasonality Decomposition for: {dtlst}\n")

for col_idx, col in enumerate(dtlst):
    print(f"\nColumn: {col}")
    
    # Prepare data - remove NaT values
    valid_dates = df[col].dropna()
    
    # Create time series by aggregating by month
    tsdata = valid_dates.to_frame(name='date')
    tsdata['year_month'] = valid_dates.dt.to_period('M')
    monthly_ts = tsdata.groupby('year_month').size()
    
    # Convert period index to timestamp for decomposition
    monthly_ts.index = monthly_ts.index.to_timestamp()
    
    print(f"  Time series length: {len(monthly_ts)} months")
    print(f"  Date range: {monthly_ts.index.min()} to {monthly_ts.index.max()}")
    
    # Check if we have enough data for decomposition (at least 2 years / 24 months)
    if len(monthly_ts) >= 24:
        print(f"  Sufficient data for seasonal decomposition\n")
        
        # Perform seasonal decomposition
        # model='additive' assumes seasonality is constant over time
        # period=12 for monthly data with yearly seasonality
        decomposition = seasonal_decompose(monthly_ts, model='additive', period=12)
        
        # Create 4 subplots for this datetime column
        fig, axes = plt.subplots(4, 1, figsize=(16, 12))
        fig.suptitle(f'{col} - Seasonal Decomposition (Additive Model)', 
                    fontsize=16, fontweight='bold', y=0.995)
        
        # 1. Original Time Series
        axes[0].plot(monthly_ts.index, monthly_ts.values, color='blue', linewidth=1.5)
        axes[0].set_ylabel('Original', fontsize=11, fontweight='bold')
        axes[0].set_title('Original Time Series (Monthly Counts)', fontsize=12)
        axes[0].grid(True, alpha=0.3)
        axes[0].tick_params(axis='x', rotation=45, labelsize=9)
        
        # 2. Trend Component
        axes[1].plot(decomposition.trend.index, decomposition.trend.values, 
                    color='red', linewidth=2)
        axes[1].set_ylabel('Trend', fontsize=11, fontweight='bold')
        axes[1].set_title('Trend Component (Long-term pattern)', fontsize=12)
        axes[1].grid(True, alpha=0.3)
        axes[1].tick_params(axis='x', rotation=45, labelsize=9)
        
        # 3. Seasonal Component
        axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, 
                    color='green', linewidth=2)
        axes[2].set_ylabel('Seasonal', fontsize=11, fontweight='bold')
        axes[2].set_title('Seasonal Component (Repeating yearly pattern)', fontsize=12)
        axes[2].grid(True, alpha=0.3)
        axes[2].tick_params(axis='x', rotation=45, labelsize=9)
        
        # 4. Residual Component
        axes[3].plot(decomposition.resid.index, decomposition.resid.values, 
                    color='orange', linewidth=1)
        axes[3].set_ylabel('Residual', fontsize=11, fontweight='bold')
        axes[3].set_title('Residual Component (Noise/Irregular)', fontsize=12)
        axes[3].set_xlabel('Time', fontsize=11, fontweight='bold')
        axes[3].grid(True, alpha=0.3)
        axes[3].tick_params(axis='x', rotation=45, labelsize=9)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print(f"\n  Decomposition Statistics for {col}:")
        print(f"     • Trend range: {decomposition.trend.min():.2f} to {decomposition.trend.max():.2f}")
        print(f"     • Seasonal amplitude: {decomposition.seasonal.max() - decomposition.seasonal.min():.2f}")
        print(f"     • Residual std dev: {decomposition.resid.std():.2f}")
        print(f"     • Model: Additive (Original = Trend + Seasonal + Residual)")
        
    else:
        print(f"  Insufficient data for decomposition (need ≥24 months, have {len(monthly_ts)})")
        print(f"  Showing simple monthly distribution instead\n")
        
        # Fallback: simple monthly bar chart
        months = valid_dates.dt.month
        monthly_counts = months.value_counts().sort_index()
        month_names = [calendar.month_abbr[int(m)] for m in monthly_counts.index]
        
        fig, ax = plt.subplots(1, 1, figsize=(14, 6))
        colors = plt.cm.viridis(monthly_counts.values / monthly_counts.values.max())
        ax.bar(month_names, monthly_counts.values, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{col} - Monthly Distribution (Insufficient data for decomposition)', 
                    fontsize=14, fontweight='bold')
        ax.set_xlabel('Month', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add count labels
        for j, (name, val) in enumerate(zip(month_names, monthly_counts.values)):
            ax.text(j, val + monthly_counts.values.max() * 0.02, f'{val:,}', 
                   ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.show()

print("\n" + "="*70)
print("Seasonality Decomposition Complete")
print("="*70)

In [0]:
# Chart 3: Heatmap (Day vs Hour)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get datetime columns
dtcol = df.select_dtypes(include=["datetime"])
dtlst = list(dtcol.columns)

print(f"Creating Heatmaps for: {dtlst}\n")

for col in dtlst:
    print(f"Processing {col}...")
    
    # Prepare data - remove NaT values
    valid_dates = df[col].dropna()
    
    # Create temporary DataFrame with hour and day of week
    hour_dow_data = valid_dates.to_frame()
    hour_dow_data['hour'] = valid_dates.dt.hour
    hour_dow_data['dow'] = valid_dates.dt.dayofweek  # 0=Monday, 6=Sunday
    
    # Create pivot table: day of week (rows) vs hour (columns)
    # Using pivot_table with aggfunc='size' to count occurrences
    heatmap_data = hour_dow_data.pivot_table(
        index='dow',          # Row axis: day of week (0-6)
        columns='hour',       # Column axis: hour (0-23)
        aggfunc='size',       # Count occurrences (frequency)
        fill_value=0          # Fill missing combinations with 0
    )
    
    dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    
    # Check if we have meaningful time data (not all midnight)
    total_records = heatmap_data.sum().sum()
    midnight_records = heatmap_data[0].sum() if 0 in heatmap_data.columns else 0
    has_time_component = (total_records > 0) and (midnight_records / total_records < 0.99)
    
    if has_time_component and len(heatmap_data.columns) > 1:
        # Create heatmap
        fig, ax = plt.subplots(1, 1, figsize=(16, 7))
        
        sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Count'}, 
                   ax=ax, linewidths=0.5, annot=False, fmt='d')
        
        ax.set_title(f'{col} - Heatmap (Day vs Hour)', fontsize=14, fontweight='bold')
        ax.set_xlabel('Hour of Day', fontsize=12)
        ax.set_ylabel('Day of Week', fontsize=12)
        ax.set_yticklabels(dow_names, rotation=0, fontsize=10)
        
        plt.tight_layout()
        plt.show()
        
        print(f"  📊 Heatmap Statistics:")
        print(f"     • Peak hour: {heatmap_data.sum().idxmax()}:00 ({heatmap_data.sum().max():,} records)")
        print(f"     • Peak day: {dow_names[heatmap_data.sum(axis=1).idxmax()]} ({heatmap_data.sum(axis=1).max():,} records)")
        print(f"     • Hours with data: {len(heatmap_data.columns)}")
        print(f"     • Records with time component: {(1 - midnight_records/total_records)*100:.1f}%\n")
    else:
        # No time component - show message
        fig, ax = plt.subplots(1, 1, figsize=(16, 7))
        ax.text(0.5, 0.5, f'No Time Component\nAll records are at 00:00:00 (midnight)\n\n{col} contains DATE only, not DATE+TIME', 
               ha='center', va='center', fontsize=16, 
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis('off')
        ax.set_title(f'{col} - Heatmap (Day vs Hour)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print(f"  ⚠️  No time component detected (all 00:00:00)\n")

print("="*70)
print("✓ Heatmap Plots Complete")
print("="*70)

In [0]:
# Chart 4: Top Years Bar Chart (Chronological Order)
import matplotlib.pyplot as plt

# Get datetime columns
dtcol = df.select_dtypes(include=["datetime"])
dtlst = list(dtcol.columns)

print(f"Creating Top Years Bar Charts for: {dtlst}\n")

for col in dtlst:
    print(f"Processing {col}...")
    
    # Prepare data - remove NaT values
    valid_dates = df[col].dropna()
    
    # Extract years and count occurrences
    years = valid_dates.dt.year
    yearly_counts = years.value_counts()
    
    # Get top 15 years by count, then sort chronologically (by year)
    top_15_by_count = yearly_counts.nlargest(15)
    top_years_chronological = top_15_by_count.sort_index(ascending=True)
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(16, 7))
    
    # Create VERTICAL bar chart with gradient colors
    colors = plt.cm.viridis(top_years_chronological.values / top_years_chronological.values.max())
    bars = ax.bar(range(len(top_years_chronological)), top_years_chronological.values, 
                   color=colors, edgecolor='black', linewidth=0.7)
    
    # Set X-axis tick labels to years (VERTICAL bars in chronological order)
    ax.set_xticks(range(len(top_years_chronological)))
    ax.set_xticklabels(top_years_chronological.index, fontsize=10, rotation=45, ha='right')
    
    # Styling for VERTICAL bars
    ax.set_title(f'{col} - Top 15 Years (Chronological Order)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add count labels on TOP of bars
    for i, (year, count) in enumerate(zip(top_years_chronological.index, top_years_chronological.values)):
        ax.text(i, count + top_years_chronological.values.max() * 0.01, 
               f'{count:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics - show highest count years AND chronological range
    print(f"  📊 Statistics for {col}:")
    print(f"\n  🏆 Highest Count Years (Top 3):")
    for rank, (year, count) in enumerate(top_15_by_count.head(3).items(), 1):
        print(f"     {rank}. {year}: {count:,} records")
    
    print(f"\n  📅 Chronological Display Range:")
    print(f"     • First year shown: {top_years_chronological.index[0]}")
    print(f"     • Last year shown: {top_years_chronological.index[-1]}")
    print(f"     • Total span: {len(top_years_chronological)} years")
    print(f"     • Top 15 total: {top_years_chronological.sum():,} records ({top_years_chronological.sum()/len(valid_dates)*100:.1f}% of all data)\n")

print("="*70)
print("✓ Top Years Bar Charts Complete")
print("="*70)

In [0]:
# Chart 5: Monthly Distribution Bar Chart
import matplotlib.pyplot as plt
import calendar

# Get datetime columns
dtcol = df.select_dtypes(include=["datetime"])
dtlst = list(dtcol.columns)

print(f"Creating Monthly Distribution Bar Charts for: {dtlst}\n")

for col in dtlst:
    print(f"Processing {col}...")
    
    # Prepare data - remove NaT values
    valid_dates = df[col].dropna()
    
    # Extract months and count occurrences
    months = valid_dates.dt.month
    monthly_counts = months.value_counts().sort_index()  # Sort by month (1-12)
    
    # Get month names
    month_names = [calendar.month_abbr[int(m)] for m in monthly_counts.index]
    
    # Create figure
    fig, ax = plt.subplots(1, 1, figsize=(16, 7))
    
    # Create VERTICAL bar chart with gradient colors
    colors = plt.cm.viridis(monthly_counts.values / monthly_counts.values.max())
    bars = ax.bar(range(len(monthly_counts)), monthly_counts.values, 
                   color=colors, edgecolor='black', linewidth=0.7)
    
    # Set X-axis tick labels to month names
    ax.set_xticks(range(len(monthly_counts)))
    ax.set_xticklabels(month_names, fontsize=11)
    
    # Styling
    ax.set_title(f'{col} - Monthly Distribution (All Months)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add count labels on TOP of bars
    for i, (month, count) in enumerate(zip(month_names, monthly_counts.values)):
        ax.text(i, count + monthly_counts.values.max() * 0.01, 
               f'{count:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"  📊 Monthly Statistics for {col}:")
    print(f"\n  🏆 Top 3 Months:")
    top_3_months = monthly_counts.nlargest(3)
    for rank, (month_num, count) in enumerate(top_3_months.items(), 1):
        month_name = calendar.month_name[int(month_num)]
        print(f"     {rank}. {month_name}: {count:,} records")
    
    print(f"\n  📉 Bottom 3 Months:")
    bottom_3_months = monthly_counts.nsmallest(3)
    for rank, (month_num, count) in enumerate(bottom_3_months.items(), 1):
        month_name = calendar.month_name[int(month_num)]
        print(f"     {rank}. {month_name}: {count:,} records")
    
    print(f"\n  📅 Distribution:")
    print(f"     • Total records: {monthly_counts.sum():,}")
    print(f"     • Average per month: {monthly_counts.mean():.0f}")
    print(f"     • Highest month: {calendar.month_name[int(monthly_counts.idxmax())]} ({monthly_counts.max():,})")
    print(f"     • Lowest month: {calendar.month_name[int(monthly_counts.idxmin())]} ({monthly_counts.min():,})")
    print(f"     • Range (max-min): {monthly_counts.max() - monthly_counts.min():,} records\n")

print("="*70)
print("✓ Monthly Distribution Bar Charts Complete")
print("="*70)

#### 4.1.d Text Variables
- Token counts, most common words, length distributions
- Histogram of text lengths
- Word cloud / bar of top words



In [0]:
# 4.1.d Text Variables Analysis
# Token counts, most common words, length distributions
# Histogram of text lengths, Word cloud / bar of top words

import re
from collections import Counter

# ── Step 1: Identify candidate text columns ──
# Object-dtype columns that are NOT datetime and NOT purely categorical codes
obj_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Remove any columns that were already converted to datetime elsewhere
dt_cols = df.select_dtypes(include=["datetime"]).columns.tolist()
text_candidates = [c for c in obj_cols if c not in dt_cols]

print(f"Object-type columns: {text_candidates}\n")

# ── Step 2: Distinguish free-text vs categorical ──
# A column is "free text" if it has many unique values AND avg word count > 1
# Categorical columns (short codes like 'R/B', 'US', 'EA') are skipped
text_cols = []
for col in text_candidates:
    non_null = df[col].dropna().astype(str)
    if len(non_null) == 0:
        continue
    n_unique = non_null.nunique()
    avg_words = non_null.str.split().str.len().mean()
    # Free text = many unique values and average more than 1 word per entry
    is_text = (n_unique > 50) and (avg_words > 1)
    print(f"  {col}: {n_unique:,} unique values, avg {avg_words:.1f} words/entry → {'TEXT' if is_text else 'categorical (skipped)'}")
    if is_text:
        text_cols.append(col)

print(f"\nIdentified free-text columns: {text_cols}\n")

# ── Step 3: If no text columns found, exit gracefully ──
if len(text_cols) == 0:
    print("=" * 60)
    print("No free-text columns found in the dataset.")
    print("All object-type columns are categorical (short codes/labels).")
    print("Text variable analysis is not applicable.")
    print("=" * 60)
else:
    # ── Step 4: Analyze each text column ──
    for col in text_cols:
        print("\n" + "=" * 60)
        print(f"TEXT COLUMN: {col}")
        print("=" * 60)

        # Get non-null text values
        text_data = df[col].dropna().astype(str)
        n_records = len(text_data)

        # ── 4a: Token (word) counts per record ──
        word_counts = text_data.str.split().str.len()
        print(f"\n--- Token Counts ---")
        print(f"  Total records: {n_records:,}")
        print(f"  Mean words/record: {word_counts.mean():.2f}")
        print(f"  Median words/record: {word_counts.median():.0f}")
        print(f"  Min words: {word_counts.min()}, Max words: {word_counts.max()}")
        print(f"  Std dev: {word_counts.std():.2f}")

        # ── 4b: Character length distributions ──
        char_lengths = text_data.str.len()
        print(f"\n--- Character Lengths ---")
        print(f"  Mean chars: {char_lengths.mean():.1f}")
        print(f"  Median chars: {char_lengths.median():.0f}")
        print(f"  Min chars: {char_lengths.min()}, Max chars: {char_lengths.max()}")

        # ── 4c: Most common words (top 20) ──
        # Tokenize all text: lowercase, split on non-alphanumeric
        all_words = []
        for text in text_data:
            words = re.findall(r"[A-Za-z0-9]+", text.lower())
            all_words.extend(words)

        word_freq = Counter(all_words)
        total_tokens = len(all_words)
        unique_tokens = len(word_freq)

        print(f"\n--- Word Frequency ---")
        print(f"  Total tokens: {total_tokens:,}")
        print(f"  Unique tokens: {unique_tokens:,}")
        print(f"  Lexical diversity (unique/total): {unique_tokens/total_tokens:.3f}")
        print(f"\n  Top 20 most common words:")
        for rank, (word, count) in enumerate(word_freq.most_common(20), 1):
            print(f"    {rank:>2}. {word:<20} {count:>6,}  ({count/total_tokens*100:.1f}%)")

        # ── 4d: Histogram of text lengths ──
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))

        # Histogram 1: Word count per record
        axes[0].hist(word_counts, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
        axes[0].set_title(f'{col} — Word Count per Record', fontsize=13, fontweight='bold')
        axes[0].set_xlabel('Number of Words')
        axes[0].set_ylabel('Frequency')
        axes[0].axvline(word_counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={word_counts.mean():.1f}')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Histogram 2: Character length per record
        axes[1].hist(char_lengths, bins=30, color='coral', edgecolor='black', alpha=0.7)
        axes[1].set_title(f'{col} — Character Length per Record', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Character Length')
        axes[1].set_ylabel('Frequency')
        axes[1].axvline(char_lengths.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={char_lengths.mean():.1f}')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # ── 4e: Word cloud OR bar chart of top words ──
        top_25_words = word_freq.most_common(25)

        if HAS_WORDCLOUD:
            # Generate word cloud
            wc_text = " ".join(all_words)
            wordcloud = WordCloud(
                width=800, height=400,
                background_color='white',
                max_words=100,
                colormap='viridis',
                relative_scaling=0.5
            ).generate(wc_text)

            fig, ax = plt.subplots(1, 1, figsize=(14, 6))
            ax.imshow(wordcloud, interpolation='bilinear')
            ax.axis('off')
            ax.set_title(f'{col} — Word Cloud (Top 100 Words)', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()

        # Bar chart of top 25 words (always shown — works without wordcloud)
        fig, ax = plt.subplots(1, 1, figsize=(14, 6))
        words_list = [w for w, c in top_25_words]
        counts_list = [c for w, c in top_25_words]
        colors = plt.cm.viridis(np.array(counts_list) / max(counts_list))
        bars = ax.barh(range(len(words_list)), counts_list, color=colors, edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(len(words_list)))
        ax.set_yticklabels(words_list, fontsize=10)
        ax.invert_yaxis()  # Most common at top
        ax.set_title(f'{col} — Top 25 Most Common Words', fontsize=14, fontweight='bold')
        ax.set_xlabel('Count')
        ax.grid(True, alpha=0.3, axis='x')
        # Add count labels
        for i, count in enumerate(counts_list):
            ax.text(count + max(counts_list) * 0.01, i, f'{count:,}', va='center', fontsize=9)
        plt.tight_layout()
        plt.show()

        # ── 4f: Summary table ──
        print(f"\n--- Summary for {col} ---")
        print(f"  Records analyzed:  {n_records:,}")
        print(f"  Total tokens:      {total_tokens:,}")
        print(f"  Unique tokens:     {unique_tokens:,}")
        print(f"  Avg words/record:  {word_counts.mean():.2f}")
        print(f"  Avg chars/record:  {char_lengths.mean():.1f}")
        print(f"  Most common word: '{word_freq.most_common(1)[0][0]}' ({word_freq.most_common(1)[0][1]:,} occurrences)")

print("\n" + "=" * 60)
print("✓ Text Variable Analysis Complete")
print("=" * 60)

### Outlier Detection - Box Plots

In [0]:
# Select only numeric columns
num_col = df.select_dtypes(include=["number"])

# Check quartiles for outlier detection
Q1 = num_col.quantile(0.25)
Q3 = num_col.quantile(0.75)
IQR = Q3 - Q1

# Define lower and upper bounds
lower_value = Q1 - 1.5 * IQR
upper_value = Q3 + 1.5 * IQR

# Identify outliers
outliers = (num_col < lower_value) | (num_col > upper_value)
outlier_counts = outliers.sum()

print("Outlier Counts by Column:")
print(outlier_counts)
print(f"\nTotal rows with outliers: {outliers.any(axis=1).sum()}")


# visulization

# Calculate number of columns and rows needed for subplots
num_cols = len(df.columns)
num_rows = (num_cols + 2) // 3  # 3 columns per row

plt.figure(figsize=(15, num_rows * 4))

for index, col in enumerate(df.columns, 1):
    plt.subplot(num_rows, 3, index)
    sns.boxplot(data=df, y=col)
    plt.title(f'Boxplot of {col}')
    plt.ylabel(col)

plt.tight_layout()
plt.show()


## 4.2 Bivariate Analysis

#### Numeric - Numeric
- Scatter plots, correlation coefficient 
- Regression line to visualize trend.
#### Numeric - Categorica
- Boxplots or violin plots grouped by category
- Compute group means/medians 
- ANOVA for significant differences
#### Categorical - Categorical
- Contingency tables, stacked bar charts 
- Chi-square test of independence
#### Time series relationships
- Lag plots, autocorrelation (ACF/PACF) 
- Rolling averages.

## 4.2.a Numeric - Numeric
- Scatter plots, correlation coefficient
- Regression line to visualize trend.

In [0]:
# Numeric - Numeric: Scatter Plots
# Visualize relationships between key numeric orbital parameters

import itertools

# Auto-detect numeric columns (exclude ID/identifier columns)
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]
print(f"Numeric columns detected: {numeric_cols}")

# Generate all unique pairs automatically
pairs = list(itertools.combinations(numeric_cols, 2))
n_pairs = len(pairs)
print(f"Total pairs: {n_pairs}\n")

# Calculate grid dimensions dynamically
n_cols = 3
n_rows = (n_pairs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows), squeeze=False)
axes = axes.flatten()

for idx, (x_col, y_col) in enumerate(pairs):
    # Filter out sentinel values (-1) and NaN
    mask = (df[x_col] != -1) & (df[y_col] != -1) & df[x_col].notna() & df[y_col].notna()
    plot_data = df[mask]
    
    axes[idx].scatter(plot_data[x_col], plot_data[y_col], alpha=0.3, s=10, c='steelblue')
    axes[idx].set_xlabel(x_col, fontsize=11)
    axes[idx].set_ylabel(y_col, fontsize=11)
    axes[idx].set_title(f'{x_col} vs {y_col}', fontsize=12, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)

# Hide unused subplots
for idx in range(n_pairs, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Scatter Plots: Numeric vs Numeric Relationships', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Correlation Coefficient
Pearson & Spearman correlation matrices for all numeric orbital parameters

In [0]:
# Numeric - Numeric: Correlation Coefficient
# Auto-detect numeric columns (exclude ID/identifier columns)
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Pearson correlation (linear relationship)
pearson_corr = df[numeric_cols].corr(method='pearson')
# Spearman correlation (monotonic relationship, robust to outliers)
spearman_corr = df[numeric_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            square=True, ax=axes[0], cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)
axes[0].set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')

sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)
axes[1].set_title('Spearman Rank Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Print strong correlations
print("=== Strong Correlations (|r| > 0.5) ===\n")
print(f"{'Pair':<45} {'Pearson':>8} {'Spearman':>8}")
print("-" * 65)
for i in range(len(pearson_corr.columns)):
    for j in range(i+1, len(pearson_corr.columns)):
        p_val = pearson_corr.iloc[i, j]
        s_val = spearman_corr.iloc[i, j]
        if abs(p_val) > 0.5 or abs(s_val) > 0.5:
            print(f"{pearson_corr.columns[i]} <-> {pearson_corr.columns[j]:<20} {p_val:>8.3f} {s_val:>8.3f}")

### Regression Line
Scatter plots with fitted regression line to visualize trends between key orbital parameter pairs

In [0]:
# Numeric - Numeric: Regression Line
from scipy.stats import linregress

# Auto-detect numeric columns (exclude ID/identifier columns)
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Auto-select top 3 pairs by strongest absolute correlation
corr_abs = df[numeric_cols].corr(method='pearson').abs()
pairs_ranked = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        pairs_ranked.append((corr_abs.iloc[i, j], numeric_cols[i], numeric_cols[j]))
pairs_ranked.sort(reverse=True)
pairs = [(x, y) for _, x, y in pairs_ranked[:3]]
print(f"Top 3 pairs by correlation: {pairs}\n")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (x_col, y_col) in enumerate(pairs):
    # Filter out sentinel values (-1) and NaN
    mask = (df[x_col] != -1) & (df[y_col] != -1) & df[x_col].notna() & df[y_col].notna()
    x_data = df.loc[mask, x_col]
    y_data = df.loc[mask, y_col]

    # Compute regression line
    slope, intercept, r_value, p_value, std_err = linregress(x_data, y_data)

    axes[idx].scatter(x_data, y_data, alpha=0.3, s=10, color='steelblue')
    axes[idx].plot(x_data, slope * x_data + intercept, color='red', linewidth=2,
                   label=f'y = {slope:.2f}x + {intercept:.1f}\nR\u00b2 = {r_value**2:.3f}')
    axes[idx].set_xlabel(x_col)
    axes[idx].set_ylabel(y_col)
    axes[idx].set_title(f'{x_col} vs {y_col}', fontsize=12, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Regression Lines: Numeric Pairs with Trend', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print regression statistics
print("\n=== Regression Statistics ===\n")
for x_col, y_col in pairs:
    mask = (df[x_col] != -1) & (df[y_col] != -1) & df[x_col].notna() & df[y_col].notna()
    slope, intercept, r_value, p_value, std_err = linregress(df.loc[mask, x_col], df.loc[mask, y_col])
    print(f"{x_col} vs {y_col}:")
    print(f"  slope: {slope:.4f}, intercept: {intercept:.2f}")
    print(f"  R\u00b2: {r_value**2:.3f}, p-value: {p_value:.2e}\n")

## 4.2.b Numeric - Categorical
- Boxplots or violin plots grouped by category
- Compute group means/medians
- ANOVA for significant differences

### Boxplots & Violin Plots
Distribution of numeric variables grouped by categorical columns

In [0]:
# Numeric - Categorical: Boxplots & Violin Plots
# Auto-detect numeric and categorical columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

catcols = df.select_dtypes(include=["object", "category"])
# Only use categorical columns with reasonable cardinality for grouping
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns (≤10 unique): {categorical_cols}\n")

n_num = len(numeric_cols)

for cat_col in categorical_cols:
    fig, axes = plt.subplots(n_num, 2, figsize=(16, 4 * n_num), squeeze=False)

    for idx, num_col in enumerate(numeric_cols):
        # Filter out sentinel values (-1)
        plot_data = df[(df[num_col] != -1) & df[num_col].notna()]

        # Boxplot
        sns.boxplot(data=plot_data, x=cat_col, y=num_col, palette='Set2', ax=axes[idx, 0])
        axes[idx, 0].set_title(f'{num_col} by {cat_col}', fontsize=11, fontweight='bold')
        axes[idx, 0].tick_params(axis='x', rotation=45)
        axes[idx, 0].grid(True, alpha=0.3, axis='y')

        # Violin plot
        sns.violinplot(data=plot_data, x=cat_col, y=num_col, palette='Set3', ax=axes[idx, 1])
        axes[idx, 1].set_title(f'{num_col} by {cat_col}', fontsize=11, fontweight='bold')
        axes[idx, 1].tick_params(axis='x', rotation=45)
        axes[idx, 1].grid(True, alpha=0.3, axis='y')

    fig.suptitle(f'Boxplots & Violin Plots: Grouped by {cat_col}', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

### Group Means & Medians
Compute group means and medians for numeric variables across categorical groups

In [0]:
# Numeric - Categorical: Group Means & Medians
# Auto-detect numeric and categorical columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

for cat_col in categorical_cols:
    print(f"\n{'='*60}")
    print(f"Grouping by: {cat_col}")
    print(f"{'='*60}")

    print(f"\n--- Mean ---")
    means = df.groupby(cat_col)[numeric_cols].mean().round(2)
    display(means)

    print(f"\n--- Median ---")
    medians = df.groupby(cat_col)[numeric_cols].median().round(2)
    display(medians)

### ANOVA
One-way ANOVA test to check if numeric variables differ significantly across categorical groups

In [0]:
# Numeric - Categorical: ANOVA
from scipy.stats import f_oneway

# Auto-detect numeric and categorical columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

for cat_col in categorical_cols:
    print(f"\n{'='*60}")
    print(f"ANOVA: Numeric Variables by {cat_col}")
    print(f"{'='*60}\n")

    for num_col in numeric_cols:
        # Filter out sentinel values (-1) and build groups
        groups = []
        for cat_val in df[cat_col].unique():
            data = df[(df[cat_col] == cat_val) & (df[num_col] != -1)][num_col].dropna()
            if len(data) > 0:
                groups.append(data)

        if len(groups) >= 2:
            f_stat, p_value = f_oneway(*groups)
            sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
            print(f"  {num_col}: F={f_stat:.2f}, p={p_value:.4e} {sig}")
        else:
            print(f"  {num_col}: Not enough groups for ANOVA")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

#### 4.2.c Categorical - Categorical
- Contingency tables, stacked bar charts 
- Chi-square test of independence


### Contingency Tables
Cross-tabulations between key categorical variables

In [0]:
# Categorical - Categorical: Contingency Tables
import itertools

# Auto-detect categorical columns (≤10 unique values)
catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

print(f"Categorical columns (≤10 unique): {categorical_cols}\n")

# Generate all unique pairs
pairs = list(itertools.combinations(categorical_cols, 2))
print(f"Total pairs: {len(pairs)}\n")

for cat1, cat2 in pairs:
    print(f"\n{'='*60}")
    print(f"Contingency Table: {cat1} vs {cat2}")
    print(f"{'='*60}")
    ct = pd.crosstab(df[cat1], df[cat2], margins=True)
    display(ct)

### Stacked Bar Charts
Visualize relationships between categorical variable pairs using stacked bar charts

In [0]:
# Categorical - Categorical: Stacked Bar Charts
import itertools

# Auto-detect categorical columns (≤10 unique values)
catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

# Generate all unique pairs
pairs = list(itertools.combinations(categorical_cols, 2))
n_pairs = len(pairs)

# Calculate grid dimensions dynamically
n_cols = 2
n_rows = (n_pairs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows), squeeze=False)
axes = axes.flatten()

for idx, (cat1, cat2) in enumerate(pairs):
    ct = pd.crosstab(df[cat1], df[cat2])
    ct.plot(kind='bar', stacked=True, ax=axes[idx], colormap='Set2', edgecolor='black')
    axes[idx].set_title(f'{cat1} vs {cat2}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel(cat1)
    axes[idx].set_ylabel('Count')
    axes[idx].legend(title=cat2, fontsize=8)
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')

# Hide unused subplots
for idx in range(n_pairs, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Stacked Bar Charts: Categorical vs Categorical', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Chi-Square Test
Chi-square test of independence to check if categorical variables are statistically associated

In [0]:
# Categorical - Categorical: Chi-Square Test
from scipy.stats import chi2_contingency
import itertools

# Auto-detect categorical columns (≤10 unique values)
catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

# Generate all unique pairs
pairs = list(itertools.combinations(categorical_cols, 2))

print("=== Chi-Square Test of Independence ===\n")
print(f"{'Pair':<45} {'Chi2':>10} {'df':>5} {'p-value':>12} {'Significant':>12}")
print("-" * 85)

for cat1, cat2 in pairs:
    ct = pd.crosstab(df[cat1], df[cat2])
    chi2, p, dof, expected = chi2_contingency(ct)
    sig = "Yes ***" if p < 0.001 else "Yes **" if p < 0.01 else "Yes *" if p < 0.05 else "No (ns)"
    print(f"{cat1} vs {cat2:<25} {chi2:>10.2f} {dof:>5} {p:>12.4e} {sig:>12}")

print("\nSignificance: *** p<0.001, ** p<0.01, * p<0.05, ns = not significant")

#### 4.2.d Time series relationships
- Lag plots, autocorrelation (ACF/PACF) 
- Rolling averages.

### Lag Plots
Lag plots to check for autocorrelation patterns in yearly counts for each datetime column

In [0]:
# Time Series: Lag Plots
from pandas.plotting import lag_plot

# Auto-detect datetime columns
dtcols = list(df.select_dtypes(include=["datetime"]).columns)
print(f"Datetime columns detected: {dtcols}\n")

for col in dtcols:
    # Build yearly time series from datetime column
    yearly_counts = df.groupby(df[col].dt.year).size()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for idx, lag in enumerate([1, 2, 3]):
        lag_plot(yearly_counts, lag=lag, ax=axes[idx], alpha=0.6, s=50, c='steelblue')
        axes[idx].set_title(f'{col} — Lag Plot (lag={lag})', fontsize=12, fontweight='bold')
        axes[idx].grid(True, alpha=0.3)

    plt.suptitle(f'Lag Plots: {col} (Yearly Counts)', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print autocorrelation at different lags
    print(f"Autocorrelation at different lags for {col}:")
    for lag in range(1, 11):
        ac = yearly_counts.autocorr(lag=lag)
        print(f"  Lag {lag}: r = {ac:.3f}")
    print()

### Autocorrelation (ACF/PACF)
ACF and PACF plots to identify autocorrelation structure in yearly counts

In [0]:
# Time Series: Autocorrelation (ACF/PACF)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Auto-detect datetime columns
dtcols = list(df.select_dtypes(include=["datetime"]).columns)

for col in dtcols:
    yearly_counts = df.groupby(df[col].dt.year).size()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    plot_acf(yearly_counts, lags=20, ax=axes[0], alpha=0.05)
    axes[0].set_title(f'ACF — {col} (Yearly Counts)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Lag')
    axes[0].grid(True, alpha=0.3)

    plot_pacf(yearly_counts, lags=20, ax=axes[1], alpha=0.05, method='ywm')
    axes[1].set_title(f'PACF — {col} (Yearly Counts)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Lag')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'Autocorrelation: {col}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

### Rolling Averages
Moving averages to smooth year-to-year fluctuations and reveal long-term trends

In [0]:
# Time Series: Rolling Averages
# Auto-detect datetime columns
dtcols = list(df.select_dtypes(include=["datetime"]).columns)

for col in dtcols:
    yearly_counts = df.groupby(df[col].dt.year).size()

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Chart 1: Rolling averages of different windows
    axes[0].plot(yearly_counts.index, yearly_counts.values, label='Raw', alpha=0.5, color='gray')
    axes[0].plot(yearly_counts.index, yearly_counts.rolling(window=3).mean(), label='3-year MA', linewidth=2, color='blue')
    axes[0].plot(yearly_counts.index, yearly_counts.rolling(window=5).mean(), label='5-year MA', linewidth=2, color='green')
    axes[0].plot(yearly_counts.index, yearly_counts.rolling(window=10).mean(), label='10-year MA', linewidth=2, color='red')
    axes[0].set_title(f'{col} — Rolling Averages', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Count')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Chart 2: Rolling mean with std dev band
    rolling_mean = yearly_counts.rolling(window=5).mean()
    rolling_std = yearly_counts.rolling(window=5).std()

    axes[1].plot(yearly_counts.index, yearly_counts.values, label='Raw', alpha=0.4, color='gray')
    axes[1].plot(rolling_mean.index, rolling_mean.values, label='5-year Rolling Mean', linewidth=2, color='blue')
    axes[1].fill_between(rolling_std.index,
                          rolling_mean - rolling_std, rolling_mean + rolling_std,
                          alpha=0.2, color='blue', label='±1 Std Dev')
    axes[1].set_title(f'{col} — Rolling Mean ± Std Dev', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Count')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(f'Rolling Averages: {col}', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print rolling statistics
    last_mean = rolling_mean.dropna().tail(1).values[0] if len(rolling_mean.dropna()) > 0 else 0
    last_std = rolling_std.dropna().tail(1).values[0] if len(rolling_std.dropna()) > 0 else 0
    print(f"{col} — Rolling Stats (5-year window):")
    print(f"  Recent mean: {last_mean:.0f}/year")
    print(f"  Recent std:  {last_std:.0f}")
    print(f"  Overall mean: {yearly_counts.mean():.0f}/year")
    print(f"  Peak: {yearly_counts.max():.0f} in {yearly_counts.idxmax()}\n")

## 4.3 Multivariate Analysis
- Heatmap of correlations (numeric). 
- Interaction plots (for interactions between two features affecting target). 
- Dimensionality reduction: PCAUMAPA-SNE to inspect clusters or structure. 
- Clustering to tind natural groups. 
- Regression/Predictive Relationships.

### Correlation Heatmap (Numeric)
Pearson & Spearman heatmaps plus hierarchical clustermap of all numeric variables

In [0]:
# Multivariate: Correlation Heatmap (Numeric)
# Auto-detect numeric columns (exclude ID/identifier columns)
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Compute correlation matrices
pearson_corr = df[numeric_cols].corr(method='pearson')
spearman_corr = df[numeric_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            square=True, ax=axes[0], cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)
axes[0].set_title('Pearson Correlation Matrix', fontsize=13, fontweight='bold')

sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.8}, vmin=-1, vmax=1)
axes[1].set_title('Spearman Rank Correlation Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

# Hierarchical clustermap
sns.clustermap(pearson_corr, annot=True, cmap='coolwarm', fmt='.2f',
               linewidths=0.5, figsize=(10, 8), vmin=-1, vmax=1)
plt.suptitle('Pearson Correlation Clustermap (Hierarchical)', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Print strong correlations
p_cols = pearson_corr.columns.tolist()
n = len(p_cols)
print("\n=== Strong Correlations (|r| > 0.5) ===\n")
print(f"{'Pair':<45} {'Pearson':>8} {'Spearman':>8}")
print("-" * 65)
for i in range(n):
    for j in range(i+1, n):
        p_val = pearson_corr.iloc[i, j]
        s_val = spearman_corr.iloc[i, j]
        if abs(p_val) > 0.5 or abs(s_val) > 0.5:
            print(f"{p_cols[i]} <-> {p_cols[j]:<20} {p_val:>8.3f} {s_val:>8.3f}")

### Interaction Plots
How two features jointly affect a target variable — regression slopes by group and two-way pivot heatmaps

In [0]:
# Multivariate: Interaction Plots
# Auto-detect numeric and categorical columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

catcols = df.select_dtypes(include=["object", "category"])
categorical_cols = [c for c in catcols.columns if df[c].nunique() <= 10]

# Auto-select target: variable with highest mean abs correlation
corr_abs = df[numeric_cols].corr(method='pearson').abs().mean().sort_values(ascending=False)
target = corr_abs.index[0]
predictors = [c for c in numeric_cols if c != target][:2]
print(f"Target: {target}")
print(f"Top predictors: {predictors}")
print(f"Categorical groups: {categorical_cols[:2]}\n")

# Plot 1: lmplot with hue (numeric × categorical interaction)
for cat_col in categorical_cols[:2]:
    for pred in predictors:
        mask = (df[pred] != -1) & (df[target] != -1) & df[pred].notna() & df[target].notna()
        plot_data = df[mask].sample(n=min(5000, mask.sum()), random_state=42)
        g = sns.lmplot(data=plot_data, x=pred, y=target, hue=cat_col,
                       height=5, aspect=1.5, scatter_kws={'alpha': 0.3, 's': 10})
        g.fig.suptitle(f'Interaction: {pred} → {target} by {cat_col}', fontsize=13, fontweight='bold', y=1.02)
        plt.show()

# Plot 2: Two-way interaction heatmap (categorical × categorical → mean target)
if len(categorical_cols) >= 2:
    cat1, cat2 = categorical_cols[0], categorical_cols[1]
    mask = (df[target] != -1) & df[target].notna()
    pivot = df[mask].pivot_table(values=target, index=cat1, columns=cat2, aggfunc='mean')
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.5)
    ax.set_title(f'Mean {target}: {cat1} × {cat2}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

### Dimensionality Reduction: PCA / t-SNE / UMAP
Project numeric features into 2D to inspect clusters or structure

In [0]:
# Multivariate: Dimensionality Reduction (PCA / t-SNE / UMAP)
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Auto-detect numeric columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Prepare data: replace sentinel (-1) with NaN, drop missing
data = df[numeric_cols].replace(-1, np.nan).dropna()
print(f"Records for dimensionality reduction: {len(data):,}")

# Scale the data (required for PCA / t-SNE)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Auto-detect categorical for coloring
catcols = df.select_dtypes(include=["object", "category"])
color_candidates = [c for c in catcols.columns if df[c].nunique() <= 10]
color_col = color_candidates[0]
colors = pd.Categorical(df.loc[data.index, color_col]).codes

# PCA
pca = PCA(n_components=min(len(numeric_cols), 3))
pca_result = pca.fit_transform(data_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.round(3)}")
print(f"Cumulative: {pca.explained_variance_ratio_.cumsum().round(3)}")

# t-SNE (subsample for speed)
sample_size = min(5000, len(data_scaled))
sample_idx = np.random.RandomState(42).choice(len(data_scaled), sample_size, replace=False)
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(data_scaled[sample_idx])
tsne_colors = pd.Categorical(df.loc[data.index[sample_idx], color_col]).codes

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# PCA scatter (PC1 vs PC2)
axes[0].scatter(pca_result[:, 0], pca_result[:, 1], c=colors, cmap='Set1', alpha=0.3, s=10)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].set_title(f'PCA (colored by {color_col})', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# t-SNE scatter
axes[1].scatter(tsne_result[:, 0], tsne_result[:, 1], c=tsne_colors, cmap='Set1', alpha=0.3, s=10)
axes[1].set_title(f't-SNE (n={sample_size}, colored by {color_col})', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# PCA scatter (PC1 vs PC3)
axes[2].scatter(pca_result[:, 0], pca_result[:, 2], c=colors, cmap='Set1', alpha=0.3, s=10)
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC3 ({pca.explained_variance_ratio_[2]*100:.1f}%)')
axes[2].set_title('PCA: PC1 vs PC3', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Try UMAP (optional — requires umap-learn)
try:
    from umap import UMAP
    umap_result = UMAP(n_components=2, random_state=42).fit_transform(data_scaled[sample_idx])
    umap_colors = pd.Categorical(df.loc[data.index[sample_idx], color_col]).codes
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))
    ax.scatter(umap_result[:, 0], umap_result[:, 1], c=umap_colors, cmap='Set1', alpha=0.3, s=10)
    ax.set_title(f'UMAP (n={sample_size}, colored by {color_col})', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("UMAP not installed. Install with: %pip install umap-learn")

# PCA loadings
pca_cols = [f'PC{i+1}' for i in range(pca.n_components_)]
loadings = pd.DataFrame(pca.components_.T, columns=pca_cols, index=numeric_cols)
print("\nPCA Loadings (feature contributions to each component):")
display(loadings.round(3))

### Clustering (KMeans)
Find natural groups in the data using KMeans, with elbow method and cluster profiles

In [0]:
# Multivariate: Clustering (KMeans)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Auto-detect numeric columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Prepare data
data = df[numeric_cols].replace(-1, np.nan).dropna()
print(f"Records for clustering: {len(data):,}")

# Scale the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Elbow method to find optimal K
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(data_scaled)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Elbow plot
axes[0].plot(list(K_range), inertias, marker='o', linewidth=2, color='steelblue')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (Within-Cluster SSE)')
axes[0].set_title('Elbow Method: Optimal K', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Apply KMeans with K=4
optimal_k = 4
km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = km.fit_predict(data_scaled)

# PCA for visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(data_scaled)

# Cluster scatter in PCA space
axes[1].scatter(pca_result[:, 0], pca_result[:, 1], c=clusters, cmap='viridis', alpha=0.3, s=10)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title(f'KMeans Clusters (K={optimal_k}) in PCA Space', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Cluster sizes bar chart
cluster_counts = pd.Series(clusters).value_counts().sort_index()
axes[2].bar(cluster_counts.index, cluster_counts.values, color='steelblue', edgecolor='black')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Count')
axes[2].set_title('Cluster Sizes', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')
for i, count in enumerate(cluster_counts.values):
    axes[2].text(i, count + cluster_counts.max() * 0.01, f'{count:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# Cluster profiles (mean of each numeric variable per cluster)
data_with_clusters = data.copy()
data_with_clusters['Cluster'] = clusters
cluster_profiles = data_with_clusters.groupby('Cluster').mean().round(2)
print("\nCluster Profiles (Mean Values):")
display(cluster_profiles)

### Regression / Predictive Relationships
Multiple linear regression to understand how numeric features jointly predict a target variable

In [0]:
# Multivariate: Regression / Predictive Relationships
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# Auto-detect numeric columns
numcols = df.select_dtypes(include=["number"])
numeric_cols = [c for c in numcols.columns if 'ID' not in c]

# Auto-select target: variable with highest mean abs correlation
corr_abs = df[numeric_cols].corr(method='pearson').abs().mean().sort_values(ascending=False)
target = corr_abs.index[0]
predictors = [c for c in numeric_cols if c != target]
print(f"Target: {target}")
print(f"Predictors: {predictors}\n")

# Prepare data (filter sentinel values)
data = df[numeric_cols].replace(-1, np.nan).dropna()
X = data[predictors]
y = data[target]

# Fit multiple linear regression
model = LinearRegression()
model.fit(X, y)
r2 = model.score(X, y)
cv_r2 = cross_val_score(model, X, y, cv=5).mean()

print(f"Multiple Linear Regression: {target} ~ {' + '.join(predictors)}")
print(f"  R² (in-sample): {r2:.3f}")
print(f"  R² (5-fold CV): {cv_r2:.3f}")
print(f"\n  Coefficients:")
for pred, coef in zip(predictors, model.coef_):
    print(f"    {pred}: {coef:.4f}")
print(f"  Intercept: {model.intercept_:.2f}")

# Actual vs Predicted + Residual plots
y_pred = model.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(y, y_pred, alpha=0.3, s=10, color='steelblue')
max_val = max(y.max(), y_pred.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel(f'Actual {target}')
axes[0].set_ylabel(f'Predicted {target}')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel(f'Predicted {target}')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residual Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Phase 5: Feature Engineering & Preprocessing

## 4. Data Normalization / Feature Scaling

#### Note: Feature Scaling Not Applied 
- Feature scaling is not applied to this project as we not going to do any Machine learning part.

### When to Use Feature Scaling (For ML Modeling)

*Rescales continuous numerical features with large variations (e.g., altitudes reaching 36,000km vs. orbital periods of 90minutes) so algorithms treat them equally.*

* **`OrbitalPeriodMinutes`**
* **`InclinationDegrees`**
* **`ApogeeKM`**
* **`PerigeeKM`**
* **`RadarCrossSectionSQM`**



#### Note: Scaling Methods for ML

* **`RobustScaler`**
* **When to use:** When data has **heavy/extreme outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \text{Median}}{\text{IQR}}$$


*Where IQR = Q3 (75th percentile) - Q1 (25th percentile)*


* **`MinMaxScaler`**
* **When to use:** When we need a **values in [0, 1] range** and have **no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}$$




* **`StandardScaler`**
* **When to use:** When data follows a **bell curve (normal distribution)** with **few or no outliers**.
* **Formula:**
$$x_{\text{scaled}} = \frac{x - \mu}{\sigma}$$


mean is $${\mu}$$ 
standard deviation is $$\sigma$$



---

### Code

```python
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

```

## 5. Data Binning (Categorical Grouping)

#### Note:

**When to Bin in Python**

* If you're building **machine learning models** → bin in Python
* If you're making **charts in Python** (matplotlib, seaborn, Plotly) → bin in Python
* If you have **fixed business rules** that never change → bin in Python

---

**When NOT to Bin (Let Power BI Do It)**

* If your **final dashboard is in Power BI** → don't bin in Python
* If you want **smaller file sizes** → don't bin in Python
* If users need to **change bin ranges** in the report → don't bin in Python
* If you need **exact numbers later** → don't bin in Python

#### Column: MaxAltitudeKM
* **-1 - 2000:** Low Earth Orbit (LEO)
* **2000 - 35785:** Medium Earth Orbit (MEO)
* **35785 - 36000:** Geostationary Earth Orbit (GEO)
* **36000 - np.inf:** High Earth Orbit (HEO)

In [0]:

# 1. Orbital Altitude Bins (ApogeeKM)
# low -1 to 2000, medium 2000 to 35785, geo 35785 to 36000, Heo 36000 to infinity (float('inf))
orbit_bins = [-1, 2000, 35785, 36000, np.inf]
orbit_labels = ['Low Earth Orbit', 'Medium Earth Orbit', 'Geostationary Orbit', 'High Earth Orbit']
df['OrbitClass'] = pd.cut(df['MaxAltitudeKM'], bins=orbit_bins, labels=orbit_labels)

df

#### Column: RadarCrossSectionSQM
* **-1 - 0.1**: Small
* **0.1 - 1.0**: Medium
* **1.0 - np.inf**: Large

In [0]:
# Column: RadarCrossSectionSQM

size_bins = [-1, 0.1, 1.0, np.inf]
size_labels = ['Small', 'Medium', 'Large']
df['RadarSize'] = pd.cut(df['RadarSizeSQM'], bins=size_bins, labels=size_labels)
df.head(4)

#### Column: LaunchDate
* **1980-1900:** *19th Century*
* **1900–2000:** *20th century*
* **2000–Present:** *21st century*

In [0]:
# Column: LaunchDate 

launch_bins = [1800, 1900, 2000, 2035]
launch_labels = ['19th Century', '20th Century', '21st Century']
df['LaunchEra'] = pd.cut(df['LaunchDate'].dt.year, bins=launch_bins, labels=launch_labels)

## 6. Creating Indicator Variables

### Binary Flags for Power BI

*Creates binary (1 or 0) flags to easily isolate specific conditions in Power BI measures or ML features.*

* **`IsDecayed`:** `1` if `DecayDate` is present (or `OrbitState == "IMP"`), `0` if still in orbit.
* **`IsActive`:** `1` if `OperationalStatus == "+"` (or `ObjectType == "PAY"` and `DecayDate` is null), `0` otherwise.
* **`IsDebris`:** `1` if `ObjectType == "DEB"`, `0` otherwise.
* **`IsRocketBody`:** `1` if `ObjectType == "R/B"`, `0` otherwise.



In [0]:
import numpy as np

# 1. IsDecayed: 1 if DecayDate is present or OrbitState is 'IMP', 0 if still in orbit
df["IsDecayed"] = np.where(df["DecayDate"].notna() | (df["OrbitState"] == "IMP"), 1, 0)

# 2. IsActive: 1 if OperationalStatus is '+' or (ObjectType is 'PAY' and DecayDate is NaT), 0 otherwise
df["IsActive"] = np.where((df["OperationalStatus"] == "+") | ((df["ObjectType"] == "PAY") & (df["DecayDate"].isna())), 1, 0)

# 3. IsDebris: 1 if ObjectType is 'DEB', 0 otherwise
df["IsDebris"] = np.where(df["ObjectType"] == "DEB", 1, 0)

# 4. IsRocketBody: 1 if ObjectType is 'R/B', 0 otherwise
df["IsRocketBody"] = np.where(df["ObjectType"] == "R/B", 1, 0)

# Display the newly created binary columns to verify
display(df[["ObjectName", "ObjectType", "IsDecayed", "IsActive", "IsDebris", "IsRocketBody"]].head(10))

In [0]:
df.to_csv("/Workspace/Users/guruvendra47@gmail.com/Space-Debris-Project/raw/space_debris_cleaned.csv", index=False)
